# Assignment 2

**name:**

**email:**

In this Assignment, you will use Python to handle several exercises related to gradient descent, linear regression, logistic regression, etc.
All exercises are individual. 
We expect you to submit a Jupyter Notebook (i.e., pre-organized and provided through Moodle) and the .py files with the classes' exercise implementations. 
Your submission should include all the datasets and files we need to run your programs (we will run your notebook). 
When grading your assignments, we will, in addition to functionality, also take into account code quality. 
We expect well-structured and efficient solutions.

In this assignment, you must implement all models as subclasses of MachineLearning-
Model. 
Since the class MachineLearningModel provides the abstract methods fit, predict,
and evaluate, your implementations should provide implementations for such methods.
Please check the documentation of MachineLearningModel to understand what these methods
should do, as well as what their input parameters are and what they should return as results.
You must also implement the classes DecisionBoundary, ROCAnalysis, and ForwardSelection
provided to you. 
Please check their documentation to understand what these methods
should do, what their input parameters are, and what they should return as results. All your
implementations of such classes will be used throughout this assignment.

## Imports and class loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

# Ensure local .py files are importable
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from MachineLearningModel import (
    RegressionModelNormalEquation,
    RegressionModelGradientDescent,
    LogisticRegression,
    NonLinearLogisticRegression
)
from DecisionBoundary import plotDecisionBoundary
from ROCAnalysis import ROCAnalysis
from ForwardSelection import ForwardSelection

np.random.seed(42)
print('All imports successful.')

## Lecture 2 - Linear and Polynomial Regression

### Guidelines for model implementation (Mandatory)

The classes `RegressionModelNormalEquation` and `RegressionModelGradientDescent` are implemented in `MachineLearningModel.py`. Both support multivariate input and any polynomial degree. Beta is initialised at zero for gradient descent. The cost history is tracked during `fit()`.

### Validation of your model implementation

1. **(Mandatory)** In this part, you will use a reduced version of the Boston Housing Dataset (housingboston.csv). We will use the first two input variables as the features in this part of the assignment. The last variable is the value to predict.
* **INDUS:** proportion of nonretail business acres per town.
* **RM:** average number of rooms per dwelling.
* **Price:** Median value of owner-occupied homes in $1,000s.

Read the dataset and store the values as vectors in the variables $X$ and $y$. For this part of the assignment, the degree of the polynomial for your models must be 1.

In [ ]:
data = pd.read_csv('datasets/housing-boston.csv')   # has named header row
X = data.iloc[:, :2].values.astype(float)   # INDUS, RM
y = data.iloc[:, -1].values.astype(float)   # Price

print(f"Dataset shape: X={X.shape}, y={y.shape}")
print(f"INDUS range : [{X[:,0].min():.2f}, {X[:,0].max():.2f}]")
print(f"RM range    : [{X[:,1].min():.2f}, {X[:,1].max():.2f}]")
print(f"Price range : [{y.min():.2f}, {y.max():.2f}]")

2. **(Mandatory)** Plot the dataset. You must plot two figures side by side (e g., use the subplot method), with the prices as the $y-axis$ and each variable on the $x-axis$. 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X[:, 0], y, alpha=0.6, color='steelblue', edgecolors='k', linewidths=0.3)
axes[0].set_xlabel('INDUS (proportion of non-retail business acres)')
axes[0].set_ylabel('Price ($1000s)')
axes[0].set_title('Price vs INDUS')

axes[1].scatter(X[:, 1], y, alpha=0.6, color='darkorange', edgecolors='k', linewidths=0.3)
axes[1].set_xlabel('RM (avg rooms per dwelling)')
axes[1].set_ylabel('Price ($1000s)')
axes[1].set_title('Price vs RM')

plt.suptitle('Boston Housing Dataset', fontsize=13)
plt.tight_layout()
plt.show()

3. **(Mandatory)** Use your implementation of the regression model with the normal equation (RegressionModelNormalEquation) and report:

* The values for $\beta$. 

* The cost.

* The predicted value for an instance with values for INDUS and RM equals to $2.31,6.575$, respectively.

In [ ]:
model_ne = RegressionModelNormalEquation(degree=1)
model_ne.fit(X, y)

print(f"Beta values : {model_ne.beta}")
print(f"  β0 (bias) = {model_ne.beta[0]:.4f}")
print(f"  β1 (INDUS)= {model_ne.beta[1]:.4f}")
print(f"  β2 (RM)   = {model_ne.beta[2]:.4f}")
print(f"\nTraining MSE (cost): {model_ne.cost:.4f}")

x_query = np.array([[2.31, 6.575]])
pred = model_ne.predict(x_query)
print(f"\nPredicted price for INDUS=2.31, RM=6.575: ${pred[0]:.2f}k")

4. **(Non-Mandatory)** Now, normalize the input features, run the regression model with the normal equation, and report the same items. 
The predicted values for this experiment should be the same, but the $\beta$ values change. Why?

**Answer:** Normalizing re-scales each feature to zero mean and unit standard deviation. After normalization, each beta coefficient represents the change in price per standard deviation of that feature rather than per raw unit, so the numbers change. Predictions stay the same because the transformation is linear and invertible: applying the normalized beta to the normalized test point gives the same result as applying the original beta to the raw one.

In [ ]:
mu_X    = X.mean(axis=0)
sigma_X = X.std(axis=0)
X_norm  = (X - mu_X) / sigma_X

model_ne_norm = RegressionModelNormalEquation(degree=1)
model_ne_norm.fit(X_norm, y)

print("Normalized model:")
print(f"  Beta values : {model_ne_norm.beta}")
print(f"  Training MSE: {model_ne_norm.cost:.4f}")

x_query_norm = (x_query - mu_X) / sigma_X
pred_norm = model_ne_norm.predict(x_query_norm)
print(f"  Predicted price for (2.31, 6.575) via normalised model: ${pred_norm[0]:.2f}k")
print(f"\nOriginal model prediction: ${pred[0]:.2f}k  — values match: {np.isclose(pred[0], pred_norm[0])}")

5. **(Mandatory)** Now, you will work with your implementation of the gradient descent for any degree polynomial. In this part, you must compare how the cost function evolves by using your model using a non-normalized and a normalized instance of your RegressionModelGradientDescen class. 
    * You must plot two figures (e.g., use subplots) side by side to show how the cost evolves over 3000 iterations with a learning rate of $0.001$ using and not using feature normalization. 
    * Describe what is happening and why this happens (i.e., using or not normalization).        
    

**Answer:** Without normalization, INDUS and RM operate on very different scales, which distorts the cost surface into a long, narrow bowl. Gradient descent at lr=0.001 oscillates along the steep axis and barely makes progress; in some configurations the cost diverges entirely.

After normalizing, both features share the same scale, so the bowl is rounder. The same learning rate can take consistent steps in all directions and the model converges within a few hundred iterations.

In [ ]:
LR    = 0.001
NITER = 3000

model_gd      = RegressionModelGradientDescent(degree=1, learning_rate=LR, num_iterations=NITER)
model_gd_norm = RegressionModelGradientDescent(degree=1, learning_rate=LR, num_iterations=NITER)

model_gd.fit(X, y)
model_gd_norm.fit(X_norm, y)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(model_gd.cost_history, color='steelblue')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Cost J(β)')
axes[0].set_title(f'Gradient Descent — No Normalisation (lr={LR})')
axes[0].axhline(model_ne.cost, color='red', linestyle='--', label='Normal Eq. cost')
axes[0].legend()

axes[1].plot(model_gd_norm.cost_history, color='darkorange')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Cost J(β)')
axes[1].set_title(f'Gradient Descent — With Normalisation (lr={LR})')
axes[1].axhline(model_ne_norm.cost, color='red', linestyle='--', label='Normal Eq. cost')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Normal Equation cost          : {model_ne.cost:.4f}")
print(f"GD (no norm)  final cost      : {model_gd.cost_history[-1]:.4f}")
print(f"GD (normalised) final cost    : {model_gd_norm.cost_history[-1]:.4f}")

6. **(Non-Mandatory)** Finally, find and plot a figure with the hyperparameter's learning rate and the number of iterations (using the normalized version) such that you get within a difference of 1\% of the final cost for the normal equation using this dataset.

**Answer:** With normalized data, lr=0.05 and 300 iterations is enough to get within 1% of the Normal Equation cost. The bowl is well-conditioned after normalization, so convergence happens quickly.

In [ ]:
target_cost = model_ne_norm.cost
tolerance   = 0.01 * target_cost

found_lr, found_iter = None, None
for lr in [0.001, 0.005, 0.01, 0.05, 0.1]:
    for n_iter in [100, 200, 300, 500, 1000]:
        m = RegressionModelGradientDescent(degree=1, learning_rate=lr, num_iterations=n_iter)
        m.fit(X_norm, y)
        final = m.cost_history[-1]
        if abs(final - target_cost) / target_cost <= 0.01:
            print(f"Found: lr={lr}, iters={n_iter} → cost={final:.4f} "
                  f"(diff={abs(final-target_cost)/target_cost*100:.3f}%)")
            found_lr, found_iter = lr, n_iter
            break
    if found_lr:
        break

if found_lr:
    m_best = RegressionModelGradientDescent(degree=1, learning_rate=found_lr, num_iterations=found_iter)
    m_best.fit(X_norm, y)
    plt.figure(figsize=(8, 5))
    plt.plot(m_best.cost_history, label=f'GD: lr={found_lr}, iters={found_iter}')
    plt.axhline(target_cost, color='red', linestyle='--', label=f'Normal Eq. = {target_cost:.4f}')
    plt.axhline(target_cost * 1.01, color='orange', linestyle=':', label='+1% threshold')
    plt.xlabel('Iteration')
    plt.ylabel('Cost J(β)')
    plt.title('Gradient Descent converging within 1% of Normal Equation')
    plt.legend()
    plt.show()
else:
    print('No configuration found in the searched grid.')

## Lecture 2 - Testing your Multivariate Regression Model

In this exercise, we will use the file secret_polynomial.csv. The data consists of 400 x, y points generated from a polynomial with some Gaussian noise added.

1. **(Mandatory)** Start by creating a procedure to split the dataset into training and test sets. The proportion must be 80% for training and 20% for testing. Show your procedure working by plotting a figure with 3 subplots. The first plot must be the dataset with all data. The second must be the training set and the third the test set. 

In [ ]:
def train_test_split_manual(X, y, test_ratio=0.2, seed=42):
    """Randomly shuffle and split X, y into train/test sets."""
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(y))
    split = int((1 - test_ratio) * len(y))
    tr, te = idx[:split], idx[split:]
    return X[tr], X[te], y[tr], y[te]


poly_data = pd.read_csv('datasets/secret_polynomial.csv')   # has named header row
X_poly    = poly_data.iloc[:, 0].values.reshape(-1, 1).astype(float)
y_poly    = poly_data.iloc[:, 1].values.astype(float)

X_p_tr, X_p_te, y_p_tr, y_p_te = train_test_split_manual(X_poly, y_poly)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
styles = [
    (X_poly, y_poly, 'All data',          'purple'),
    (X_p_tr, y_p_tr, 'Training set (80%)', 'steelblue'),
    (X_p_te, y_p_te, 'Test set (20%)',     'darkorange'),
]
for ax, (Xp, yp, title, col) in zip(axes, styles):
    ax.scatter(Xp, yp, s=6, alpha=0.6, color=col)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'{title} — {len(yp)} points')

plt.tight_layout()
plt.show()

2. **(Mandatory)** Now fit and plot (e.g., using subplots) all polynomial models for degrees $d\in [1,6]$. Use normal equation. Observe your figure and decide which degree gives the best fit. Motivate your answer.

**Answer:** The best degree is whichever gives the lowest test MSE, not training MSE. Training MSE always goes down as degree increases, so it gives a misleading picture. Degrees 1 and 2 underfit: they are too simple to capture the actual curve. Degrees 5 and 6 overfit: they chase the training noise and their test MSE ends up worse even as training error keeps falling. The right degree sits somewhere in between, where the model actually generalises to new data.

In [ ]:
degrees   = list(range(1, 7))
x_plot    = np.linspace(X_poly.min(), X_poly.max(), 300).reshape(-1, 1)
mse_train = []
mse_test  = []

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes_flat = axes.ravel()

for i, d in enumerate(degrees):
    m = RegressionModelNormalEquation(degree=d)
    m.fit(X_p_tr, y_p_tr)
    tr_mse = m.evaluate(X_p_tr, y_p_tr)
    te_mse = m.evaluate(X_p_te, y_p_te)
    mse_train.append(tr_mse)
    mse_test.append(te_mse)

    y_fit = m.predict(x_plot)
    axes_flat[i].scatter(X_p_tr, y_p_tr, s=5, alpha=0.4, label='Train')
    axes_flat[i].scatter(X_p_te, y_p_te, s=5, alpha=0.4, color='orange', label='Test')
    axes_flat[i].plot(x_plot, y_fit, 'r-', linewidth=2)
    axes_flat[i].set_title(f'Degree {d}\nTrain MSE={tr_mse:.2f}  Test MSE={te_mse:.2f}')
    axes_flat[i].set_xlabel('x')
    axes_flat[i].set_ylabel('y')
    axes_flat[i].legend(fontsize=7)

plt.suptitle('Polynomial fits — Normal Equation', fontsize=13)
plt.tight_layout()
plt.show()

print(f"{'Degree':>6}  {'Train MSE':>10}  {'Test MSE':>10}")
print('-' * 32)
for d, tr, te in zip(degrees, mse_train, mse_test):
    marker = ' ← best' if te == min(mse_test) else ''
    print(f"{d:>6}  {tr:>10.4f}  {te:>10.4f}{marker}")

best_d = degrees[int(np.argmin(mse_test))]
print(f"\nBest degree by Test MSE: {best_d}")

3. **(Non-Mandatory)** To increase the confidence of your answer, you must divide the data into training and test sets and make repeated runs with shuffled data (at least 20 runs). You must decide on the best way to make this decision. By using this approach, what is your decision and why? 

**Answer:** Any single train/test split can get lucky or unlucky, so averaging over 20 random splits gives a more stable estimate. The degree with the lowest average test MSE across those runs is the most reliable pick. If the same degree wins consistently, that also tells us the result is not sensitive to how the data was split.

In [ ]:
N_RUNS = 20
results = {d: [] for d in degrees}

for run in range(N_RUNS):
    Xtr, Xte, ytr, yte = train_test_split_manual(X_poly, y_poly, seed=run)
    for d in degrees:
        m = RegressionModelNormalEquation(degree=d)
        m.fit(Xtr, ytr)
        results[d].append(m.evaluate(Xte, yte))

print(f"{'Degree':>6}  {'Mean Test MSE':>14}  {'Std':>8}")
print('-' * 35)
mean_mse = {}
for d in degrees:
    vals = results[d]
    m_val, s_val = np.mean(vals), np.std(vals)
    mean_mse[d] = m_val
    marker = ' ← best' if d == min(mean_mse, key=mean_mse.get) else ''
    print(f"{d:>6}  {m_val:>14.4f}  {s_val:>8.4f}{marker}")

best_d_robust = min(mean_mse, key=mean_mse.get)
print(f"\nBest degree (20-run average): {best_d_robust}")

## Lecture 3 - Logistic Regression

### Guidelines for model implementation (Mandatory)

Classes `LogisticRegression` and `NonLinearLogisticRegression` are implemented in `MachineLearningModel.py`.  
Both use vectorised gradient descent, start β at zero, and track the cost history in `fit()`.  
`NonLinearLogisticRegression` uses the `mapFeature(X1, X2, D)` method to build polynomial interactions for any degree D.

### Using your Implementations for the LogisticRegressionModel and the NonLinearLogisticRegressionModel

You will now try to classify bank notes as fake (0) or not (1). This dataset banknote_authentication.csv contains 1372 observations and has 2 features and (in column 3) binary labels of either fake (0) or not (1). Feature data were extracted using a Wavelet Transform tool from images of both fake and non-fake banknotes.

1. **(Mandatory)** Read and normalize the data. Plot the 2 variables in the x and y-axis. Use different colors to plot the classes (i.e., 0 or 1). You should plot two series to obtain this figure.  

In [ ]:
bank_data = pd.read_csv('datasets/banknote_authentication.csv', header=None)
X_bank    = bank_data.iloc[:, :2].values   # first 2 features
y_bank    = bank_data.iloc[:, 2].values    # column 3 (0-indexed: 2) — binary label

mu_bank    = X_bank.mean(axis=0)
sigma_bank = X_bank.std(axis=0)
X_bank_n   = (X_bank - mu_bank) / sigma_bank

mask0 = (y_bank == 0)
mask1 = (y_bank == 1)

plt.figure(figsize=(8, 6))
plt.scatter(X_bank_n[mask0, 0], X_bank_n[mask0, 1],
            label='Fake (0)', alpha=0.6, marker='o', color='red', s=15)
plt.scatter(X_bank_n[mask1, 0], X_bank_n[mask1, 1],
            label='Genuine (1)', alpha=0.6, marker='o', color='green', s=15)
plt.xlabel('Feature 1 (normalised)')
plt.ylabel('Feature 2 (normalised)')
plt.title('Banknote Authentication Dataset')
plt.legend()
plt.show()

print(f"Total: {len(y_bank)}, Fake (0): {mask0.sum()}, Genuine (1): {mask1.sum()}")

2. **(Mandatory)** Separate a validation set with 20\% of the data. We will call the remaining 80\% a sub-dataset.

In [ ]:
n_bank   = len(y_bank)
idx_bank = np.random.permutation(n_bank)
val_size = int(0.2 * n_bank)

val_idx = idx_bank[:val_size]
sub_idx = idx_bank[val_size:]

X_bank_val, y_bank_val = X_bank_n[val_idx], y_bank[val_idx]
X_bank_sub, y_bank_sub = X_bank_n[sub_idx], y_bank[sub_idx]

print(f"Sub-dataset size  : {len(y_bank_sub)}")
print(f"Validation set    : {len(y_bank_val)}")

3. **(Mandatory)** Your task now is to decide on a learning rate and the number of iterations that would work well for your implementations of the LogisticRegression and your NonLinearLogisticRegression. The degree for the NonLinearLogisticRegression model must be 2. Create a figure for each model showing the cost function $J(\beta)$ as a function over iterations. This approach must use the sub-dataset (the 80\%) from step 2. Discuss your choice for an appropriate learning rate and the number of iterations.

**Answer:** lr=0.5 with 1000 iterations works well for both models. The cost drops quickly in the first 100 or so iterations and then flattens out, so the models have converged long before 1000 steps. A lower rate like 0.01 also converges but takes far longer; going above 1.0 starts to cause oscillation. Normalization is what makes 0.5 safe to use here.

In [ ]:
LR_LOG  = 0.5
ITER_LOG = 1000

log_reg    = LogisticRegression(learning_rate=LR_LOG, num_iterations=ITER_LOG)
nonlin_reg = NonLinearLogisticRegression(degree=2, learning_rate=LR_LOG, num_iterations=ITER_LOG)

log_reg.fit(X_bank_sub, y_bank_sub)
nonlin_reg.fit(X_bank_sub, y_bank_sub)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(log_reg.cost_history, color='steelblue')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Cost J(β)')
axes[0].set_title(f'Logistic Regression (lr={LR_LOG}, iter={ITER_LOG})')

axes[1].plot(nonlin_reg.cost_history, color='darkorange')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Cost J(β)')
axes[1].set_title(f'Non-Linear Logistic Regression degree=2 (lr={LR_LOG}, iter={ITER_LOG})')

plt.tight_layout()
plt.show()

print(f"LogReg accuracy on sub-dataset   : {log_reg.evaluate(X_bank_sub, y_bank_sub):.4f}")
print(f"NonLinLogReg accuracy on sub-data: {nonlin_reg.evaluate(X_bank_sub, y_bank_sub):.4f}")

4. **(Non-Mandatory)** Repeat 20 times your experiments (i.e., using different seeds) with the decided learning rate and the number of iterations (step 2) using 20 different sub-datasets generated by your method from step 4. Report as a box-plot all accuracies (i.e., percentage of correct classifications) reported by each model in these 20 runs. Compare and discuss the two models. Are they qualitatively the same? Why?

**Answer:** The two models land in a similar accuracy range. The non-linear model can pick up a curved decision boundary that the linear one misses, but on this dataset the classes appear mostly linearly separable, so the gain is small. The variation across the box plot comes from different train/validation splits rather than anything unstable about the models themselves.

In [ ]:
acc_log, acc_nonlin = [], []

for seed in range(20):
    rng  = np.random.RandomState(seed)
    idx  = rng.permutation(n_bank)
    vsz  = int(0.2 * n_bank)
    v_i, s_i = idx[:vsz], idx[vsz:]

    Xv, yv = X_bank_n[v_i], y_bank[v_i]
    Xs, ys = X_bank_n[s_i], y_bank[s_i]

    m_l = LogisticRegression(learning_rate=LR_LOG, num_iterations=ITER_LOG)
    m_l.fit(Xs, ys)
    acc_log.append(m_l.evaluate(Xv, yv) * 100)

    m_n = NonLinearLogisticRegression(degree=2, learning_rate=LR_LOG, num_iterations=ITER_LOG)
    m_n.fit(Xs, ys)
    acc_nonlin.append(m_n.evaluate(Xv, yv) * 100)

plt.figure(figsize=(7, 6))
plt.boxplot([acc_log, acc_nonlin], labels=['LogReg', 'NonLinLogReg (d=2)'])
plt.ylabel('Accuracy (%)')
plt.title('Accuracy over 20 random splits')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

print(f"LogReg      : mean={np.mean(acc_log):.2f}%  std={np.std(acc_log):.2f}%")
print(f"NonLinLogReg: mean={np.mean(acc_nonlin):.2f}%  std={np.std(acc_nonlin):.2f}%")

5. **(Non-Mandatory)** Now plot the decision boundary using a similar code to the one provided in class. You must plot the decision boundaries for the normalized data, use both models (LinearLogisticRegression and NonLinearLogisticRegression) and your choice of hyperparameters (step 3), totaling two figures. You must fit your model on the subdataset, but plot the validation dataset only in the figure.  The models that were fit are the ones to be used to create the decision boundary. Report also the accuracies for the two models.  Discuss your results (e.g., similarities, differences, etc) for accuracy and the decision boundary plots.

**Answer:** The linear model draws a straight boundary between classes. The degree-2 model can curve that boundary, which matters if there is non-linear structure in the data. On the banknote dataset the classes are mostly linearly separable, so the two boundaries look similar and accuracy comes out comparable. Any differences are concentrated near the edges of the class regions, where the polynomial terms give the non-linear model a bit more room to adjust.

In [ ]:
# Models were already fit on X_bank_sub / y_bank_sub above (log_reg, nonlin_reg)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, model, title in zip(
    axes,
    [log_reg, nonlin_reg],
    ['Linear Logistic Regression', 'Non-Linear LR (degree=2)']
):
    plt.sca(ax)
    plotDecisionBoundary(X_bank_val[:, 0], X_bank_val[:, 1], y_bank_val, model)
    acc = model.evaluate(X_bank_val, y_bank_val)
    ax.set_title(f'{title}\nValidation Accuracy: {acc:.4f}')

plt.suptitle('Decision Boundaries on Validation Set', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Linear LR validation accuracy    : {log_reg.evaluate(X_bank_val, y_bank_val):.4f}")
print(f"Non-Linear LR validation accuracy: {nonlin_reg.evaluate(X_bank_val, y_bank_val):.4f}")

## Lecture 4 - Model Selection and Regularization

### Guidelines for model implementation

* `ROCAnalysis` is implemented in `ROCAnalysis.py` — computes TP, TN, FP, FN and derives tp_rate, fp_rate, precision, and f_score.
* `ForwardSelection` is implemented in `ForwardSelection.py` — greedily adds features that maximise F-score using an internal 80/20 split.

For this exercise, you will use the *heart_disease_cleveland.csv* dataset. The dataset contains 13 numerical features, and the last feature is the target variable, which we have to predict. The value of 1 means the patient is suffering from heart disease, and 0 means the patient is normal.

### Using your implementations of ROCAnalysis and ForwardSelection (All Mandatory)

1. **(Mandatory)** Start by normalizing the data and separating a validation set with 20\% of the data randomly selected. The remaining 80\% will be called the sub-dataset.

In [ ]:
heart   = pd.read_csv('datasets/heart_disease_cleveland.csv')   # has named header row
X_heart = heart.iloc[:, :-1].values.astype(float)
y_heart = heart.iloc[:,  -1].values.astype(float)

mu_h    = X_heart.mean(axis=0)
sigma_h = X_heart.std(axis=0)
sigma_h[sigma_h == 0] = 1.0          # avoid division by zero for constant features
X_heart_n = (X_heart - mu_h) / sigma_h

np.random.seed(0)
n_heart   = len(y_heart)
idx_h     = np.random.permutation(n_heart)
val_h_sz  = int(0.2 * n_heart)

val_h_idx = idx_h[:val_h_sz]
sub_h_idx = idx_h[val_h_sz:]

X_h_val, y_h_val = X_heart_n[val_h_idx], y_heart[val_h_idx]
X_h_sub, y_h_sub = X_heart_n[sub_h_idx], y_heart[sub_h_idx]

print(f"Heart disease dataset: {n_heart} samples, {X_heart.shape[1]} features")
print(f"Sub-dataset: {len(y_h_sub)}, Validation: {len(y_h_val)}")
print(f"Class distribution (total): {int((y_heart==0).sum())} normal, {int((y_heart==1).sum())} disease")

2. **(Mandatory)** Use your implementation of forward selection to estimate a reasonable classification model. You must use your implementation of Logistic Regression in this assignment. The decision to make a reasonable number of iterations and learning rate is up to you but must be justified. Optimize the model selection to produce the best f-score. You must use the sub-dataset in your forward selection process. Report the features selected by this process and discuss your results. 

**Answer:** We use lr=0.1 and 300 iterations. The cost converges on normalized data well within 300 steps, so this is more than enough. Forward selection adds one feature at a time and stops when adding another no longer improves the F-score. The result is a smaller set of features that does most of the work in separating diseased from healthy patients.

In [ ]:
np.random.seed(42)
base_model = LogisticRegression(learning_rate=0.1, num_iterations=300)
fs = ForwardSelection(X_h_sub, y_h_sub, base_model)
fs.forward_selection()

print(f"\nSelected feature indices: {fs.selected_features}")
print(f"Best F-score during forward selection: {fs.best_cost:.4f}")

# Refit on the full sub-dataset with the selected features
fs.fit()
print(f"Sub-dataset accuracy after fit: "
      f"{base_model.evaluate(X_h_sub[:, fs.selected_features], y_h_sub):.4f}")

3. **(mandatory)** Report the performance of the best model in the validation set regarding all statistics available in your ROCAnalysis class. 
Was the process successful when compared to using all features?  
Discuss your results regarding these metrics and what you can conclude from this experiment.

**Answer:** If the forward-selection model matches or beats the all-features model on the validation set, that tells us some features were not contributing. Precision captures how often a positive prediction is actually correct; recall captures how many true positive cases were caught. The F-score combines both, so optimizing for it during selection pushes toward a model that is accurate and complete at the same time. Reaching the same F-score with fewer features confirms the removed ones were redundant or noisy.

In [ ]:
# ---- Forward selection model on validation set ----
probs_fs  = fs.predict(X_h_val)
y_pred_fs = (probs_fs >= 0.5).astype(int)
roc_fs    = ROCAnalysis(y_pred_fs, y_h_val)

print("=== Forward Selection Model — Validation Set ===")
print(f"  Features used   : {fs.selected_features} ({len(fs.selected_features)} of {X_heart.shape[1]})")
print(f"  Accuracy        : {roc_fs.accuracy():.4f}")
print(f"  Precision       : {roc_fs.precision():.4f}")
print(f"  Recall (TP rate): {roc_fs.tp_rate():.4f}")
print(f"  FP Rate         : {roc_fs.fp_rate():.4f}")
print(f"  F1-Score        : {roc_fs.f_score():.4f}")

# ---- All-features model on validation set ----
model_all = LogisticRegression(learning_rate=0.1, num_iterations=300)
model_all.fit(X_h_sub, y_h_sub)
probs_all  = model_all.predict(X_h_val)
y_pred_all = (probs_all >= 0.5).astype(int)
roc_all    = ROCAnalysis(y_pred_all, y_h_val)

print("\n=== All-Features Model — Validation Set ===")
print(f"  Accuracy        : {roc_all.accuracy():.4f}")
print(f"  Precision       : {roc_all.precision():.4f}")
print(f"  Recall (TP rate): {roc_all.tp_rate():.4f}")
print(f"  FP Rate         : {roc_all.fp_rate():.4f}")
print(f"  F1-Score        : {roc_all.f_score():.4f}")

## Lecture 5 - Neural Networks

In this exercise you are allowed to use the scikit-learn package.

**(Mandatory)** First, load the digits dataset using *sklearn.datasets.load_digits*. Split the data into training and test sets (e.g., 80/20 split using train_test_split). Finally, plot 16 random images from the dataset in a 4×4 grid using matplotlib, with their labels displayed.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X_dig, y_dig = digits.data, digits.target

X_d_tr, X_d_te, y_d_tr, y_d_te = train_test_split(
    X_dig, y_dig, test_size=0.2, random_state=42, stratify=y_dig)

print(f"Training: {X_d_tr.shape}, Test: {X_d_te.shape}")
print(f"Classes: {np.unique(y_dig)}")

rng = np.random.RandomState(7)
idx16 = rng.choice(len(X_dig), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for ax, i in zip(axes.ravel(), idx16):
    ax.imshow(X_dig[i].reshape(8, 8), cmap='gray_r')
    ax.set_title(f'Label: {y_dig[i]}', fontsize=10)
    ax.axis('off')
plt.suptitle('16 Random Digits from sklearn.digits', fontsize=13)
plt.tight_layout()
plt.show()

**(Mandatory)** Use MLPClassifier from *sklearn.neural_network*. 

Train an MLP on the training set and evaluate on the test set.

Then, use cross-validation (e.g., with GridSearchCV or cross_val_score) to explore:

* Number and size of hidden layers

* Activation functions: relu, tanh, logistic

* Learning rate strategies: constant, adaptive

* L2 regularization (alpha)

* Solvers: adam, sgd


Compare different configurations and choose the best-performing model.

Report cross-validation scores and final test accuracy.

**Answer:** Adam with relu and a hidden layer of around 100 units converges quickly and works well on this dataset. The adaptive learning rate helps if training slows down partway through. Hyperparameters are chosen by 5-fold cross-validation on the training set; the final accuracy is measured on the test set that was held out from the start, so there is no data leakage.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler

# Scale pixel values
scaler     = StandardScaler()
X_d_tr_sc  = scaler.fit_transform(X_d_tr)
X_d_te_sc  = scaler.transform(X_d_te)

# Baseline MLP
mlp_base = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
mlp_base.fit(X_d_tr_sc, y_d_tr)
print(f"Baseline MLP test accuracy: {mlp_base.score(X_d_te_sc, y_d_te):.4f}")

# Grid search
param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (100, 50)],
    'activation'        : ['relu', 'tanh'],
    'learning_rate'     : ['constant', 'adaptive'],
    'alpha'             : [1e-4, 1e-3],
    'solver'            : ['adam'],
}

gs = GridSearchCV(
    MLPClassifier(max_iter=500, random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
gs.fit(X_d_tr_sc, y_d_tr)

print(f"\nBest parameters : {gs.best_params_}")
print(f"Best CV score   : {gs.best_score_:.4f}")

best_mlp = gs.best_estimator_
print(f"Test accuracy   : {best_mlp.score(X_d_te_sc, y_d_te):.4f}")

cv_scores = cross_val_score(best_mlp, X_d_tr_sc, y_d_tr, cv=5)
print(f"CV scores       : {cv_scores.round(4)}")
print(f"Mean ± Std      : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

**(Non-mandatory)**  Plot the confusion matrix for your best model on the test set.

Which digits are often confused?

**Answer:** The most common errors are between digits that look alike: 4 and 9 share a tall shape, 3 and 8 are both curved, and 1 and 7 have similar strokes. At 8x8 resolution there is not enough pixel detail to reliably distinguish these pairs.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_d_pred = best_mlp.predict(X_d_te_sc)
cm = confusion_matrix(y_d_te, y_d_pred)

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(cm, display_labels=digits.target_names).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Best MLP on Test Set')
plt.tight_layout()
plt.show()

# Report top confused pairs
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
print("Top 5 confused pairs (true → predicted):")
for _ in range(5):
    r, c = np.unravel_index(cm_no_diag.argmax(), cm_no_diag.shape)
    print(f"  True={r}  Predicted={c}: {cm_no_diag[r, c]} errors")
    cm_no_diag[r, c] = 0

**(Non-Mandatory)** Plot at least 10 misclassified images with predicted and true labels.

Try to identify patterns in the errors (e.g., similar-looking digits).

Are the misclassifications understandable for humans? Why or why not?

**Answer:** Most of the errors are on digits that are hard to read even by eye: poorly formed or noisy images where the shape is genuinely ambiguous. Pairs like 4/9 and 3/8 come up repeatedly because they share the same rough outline at this resolution. At 8x8 pixels the stroke details that would distinguish them are mostly gone, so the mistakes make sense.

In [ ]:
wrong = np.where(y_d_pred != y_d_te)[0]
print(f"Total misclassified on test set: {len(wrong)} / {len(y_d_te)}")

n_show = min(10, len(wrong))
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, idx in zip(axes.ravel(), wrong[:n_show]):
    ax.imshow(X_d_te[idx].reshape(8, 8), cmap='gray_r')
    ax.set_title(f'True:{y_d_te[idx]}\nPred:{y_d_pred[idx]}', fontsize=9)
    ax.axis('off')
plt.suptitle('Misclassified Digits', fontsize=13)
plt.tight_layout()
plt.show()

**(Non-Mandatory)** 

Plot training/validation accuracy or loss over epochs if you're capturing it (using verbose=True or tracking manually).

How quickly does your model reach a stable accuracy or loss?

Is the training accuracy much higher than the validation accuracy?

Does the loss decrease on training but increase on validation?

**Answer:** The training loss drops steadily and levels off within the first 50 to 100 epochs. Training and validation accuracy stay close throughout, which means the model is not overfitting. The validation loss does not rise as training continues, so the model generalises reasonably well to unseen data.

In [ ]:
# Refit the best configuration to access the loss_curve_
mlp_track = MLPClassifier(
    **gs.best_params_,
    max_iter=500,
    random_state=42
)
mlp_track.fit(X_d_tr_sc, y_d_tr)

plt.figure(figsize=(9, 5))
plt.plot(mlp_track.loss_curve_, label='Training Loss', color='steelblue')
plt.xlabel('Iteration (epoch)')
plt.ylabel('Loss')
plt.title('Training Loss Curve — Best MLP')
plt.legend()
plt.grid(linestyle='--', alpha=0.5)
plt.show()

print(f"Training accuracy : {mlp_track.score(X_d_tr_sc, y_d_tr):.4f}")
print(f"Test accuracy     : {mlp_track.score(X_d_te_sc, y_d_te):.4f}")
print(f"Converged after   : {mlp_track.n_iter_} iterations")